# Notebook 11 — Empty Edit Control: Does Edit B's Success Even Matter?
**CS 590NN | Amogh | Apr 25**

## The puzzle from NB10

NB10 fit `A_displaced = 0.532 + 0.261 × B_installed`. The intercept (0.53) was the dominant term — A loses ~53% of its mass even when B totally fails.

**Hypothesis:** The mere act of running gradient steps for B perturbs A, regardless of B's success.

## 4-condition causal test

- **C0 (skip):** Edit A → measure A. (Baseline, no edit B.)
- **C1 (real):** Edit A → real edit B → measure A.
- **C2 (dry_run):** Edit A → run B's gradients with lr~0 → measure A.
- **C3 (random_target):** Edit A → edit B against a random token → measure A.

If C2 ≈ C0: parameter updates are required. If C2 >> C0: even gradient computation damages A — strongest novel finding.

## Compute

~15 min on A100. Auto-fetches pair indices from the team repo.

### 11.0 Install (run once, restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Restarting...'); os.kill(os.getpid(), 9)

### 11.1 Imports

In [ ]:
import torch, json, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb11'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

### 11.2 Load model

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f'Loaded {MODEL_NAME}: {model.cfg.n_layers}L x {model.cfg.n_heads}H')

### 11.3 Auto-fetch v3 pair indices from GitHub + load CounterFact

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main/circuit_pipeline/results'
V3_URL = f'{REPO_RAW}/sequential_editing_v3/sequential_edit_rows_v3_20260424_220951.json'

print('Fetching v3 pair indices from GitHub...')
v3_data = requests.get(V3_URL, timeout=60).json()
v3_ok = [r for r in v3_data['rows'] if r.get('status') == 'ok']

# First 10 unique pairs (mix of conditions)
seen = set()
PAIR_INDICES = []
for r in v3_ok:
    pid = r['pair_idx']
    if pid in seen: continue
    seen.add(pid)
    PAIR_INDICES.append((r['A_idx'], r['B_idx']))
    if len(PAIR_INDICES) >= 10: break
print(f'Selected {len(PAIR_INDICES)} pair indices from v3')

@dataclass
class EditSample:
    idx: int; prompt: str; subject: str
    target_new: str; target_true: str

print('\nDownloading CounterFact...')
raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()
def make_sample(idx):
    rr = raw[idx]['requested_rewrite']
    return EditSample(idx=idx, prompt=rr['prompt'].format(rr['subject']),
                      subject=rr['subject'],
                      target_new=' '+rr['target_new']['str'], target_true=' '+rr['target_true']['str'])

pairs = [{'A': make_sample(a), 'B': make_sample(b)} for a, b in PAIR_INDICES]
print(f'Built {len(pairs)} pairs')

### 11.4 ROME-trace localization

In [ ]:
def rome_trace_top_layers(model, sample, k=5):
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0].item()
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0].item()
    tokens = model.to_tokens(sample.prompt)
    n_tok = tokens.shape[1]
    if n_tok <= 2: return list(range(min(k, model.cfg.n_layers)))
    subj_pos = list(range(1, n_tok-1))
    corrupt = tokens.clone()
    corrupt[0, subj_pos] = torch.randint(1000, model.cfg.d_vocab-1, (len(subj_pos),))
    with torch.no_grad():
        cl, cc = model.run_with_cache(tokens)
        kl, _ = model.run_with_cache(corrupt)
    clean_score = (cl[0,-1,true_id] - cl[0,-1,new_id]).item()
    corrupt_score = (kl[0,-1,true_id] - kl[0,-1,new_id]).item()
    eff = max(abs(clean_score - corrupt_score), 0.5)
    effects = []
    for L in range(model.cfg.n_layers):
        hn = f'blocks.{L}.hook_mlp_out'
        if hn not in cc: continue
        ca = cc[hn].clone()
        def hk(v, hook): return ca
        with torch.no_grad():
            patched = model.run_with_hooks(corrupt, fwd_hooks=[(hn, hk)])
        recovery = (patched[0,-1,true_id] - patched[0,-1,new_id]).item() - corrupt_score
        effects.append((L, abs(recovery)/eff))
    effects.sort(key=lambda x: -x[1])
    del cc; torch.cuda.empty_cache()
    return [L for L, _ in effects[:k]]

### 11.5 Edit function with mode parameter

In [ ]:
NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The square root of nine is', 'Ten times ten equals',
    'The capital of Japan is', 'The largest ocean on Earth is the',
    'Mount Everest is located in the', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
    'Plants produce oxygen through a process called', 'The Earth orbits the',
    'A week contains seven', 'The primary colors are red, blue, and',
    'Humans have two lungs and one', 'A triangle has three',
]

def cache_ref(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            t = model.to_tokens(p)
            ref_lp = torch.log_softmax(model(t)[0,-1,:], dim=-1).detach().clone()
            cache.append((t, ref_lp))
    return cache

def kl_against(model, ref_cache):
    total = 0.0
    for t, ref_lp in ref_cache:
        edit_lp = torch.nn.functional.log_softmax(model(t)[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def get_params(model, mlp_layers):
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    return params

def edit(model, sample, mlp_layers, n_steps, lr, beta_kl, ref_cache, mode='real', random_seed=0):
    if mode == 'skip':
        return
    params = get_params(model, mlp_layers)
    if not params:
        params = get_params(model, list(range(model.cfg.n_layers)))
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    eff_lr = lr if mode != 'dry_run' else 1e-12
    opt = torch.optim.Adam(params, lr=eff_lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    if mode == 'random_target':
        random.seed(random_seed)
        target_id = torch.tensor(random.randint(1000, model.cfg.d_vocab - 1000), device=DEVICE)
    else:
        target_id = new_id
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[target_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_against(model, ref_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        return float(torch.softmax(model(model.to_tokens(sample.prompt))[0,-1,:], dim=-1)[new_id])

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    torch.cuda.empty_cache()

print('Snapshotting weights...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f'GPU free: {free/1e9:.1f} GB')

### 11.6 The 4-condition loop

In [ ]:
CONDITIONS = ['skip', 'real', 'dry_run', 'random_target']
N_STEPS_A = 5
N_STEPS_B = 3
BETA_KL = 0.1
LR = 5e-3

ROWS_FILE = RESULTS_DIR / f'empty_edit_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
rows = []
started = datetime.now()
ref_cache = cache_ref(model, NEUTRAL)

all_samples = {s.idx: s for p in pairs for s in (p['A'], p['B'])}
print(f'Pre-computing ROME-trace for {len(all_samples)} unique samples...')
circuits = {idx: rome_trace_top_layers(model, s, k=5) for idx, s in all_samples.items()}
print('Done.')

total_runs = len(pairs) * len(CONDITIONS)
print(f'Total runs: {total_runs}')

run_idx = 0
for p_idx, p in enumerate(pairs):
    A, B = p['A'], p['B']
    for cond in CONDITIONS:
        run_idx += 1
        try:
            restore(model, original_state)
            edit(model, A, circuits[A.idx], n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL,
                 ref_cache=ref_cache, mode='real')
            p_A_postA = measure_p_new(model, A)
            edit(model, B, circuits[B.idx], n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL,
                 ref_cache=ref_cache, mode=cond, random_seed=p_idx)
            p_A_after = measure_p_new(model, A)
            p_B_after = measure_p_new(model, B)
            A_displaced = max(0, p_A_postA - p_A_after)
            row = {
                'pair_idx': p_idx, 'A_idx': A.idx, 'B_idx': B.idx, 'condition': cond,
                'p_A_postA': p_A_postA, 'p_A_after_B': p_A_after, 'p_B_after_B': p_B_after,
                'A_displaced': A_displaced, 'status': 'ok',
            }
            rows.append(row)
            print(f'[{run_idx:>2}/{total_runs}] pair {p_idx:>2}  cond={cond:>13}  '
                  f'p_A_postA={p_A_postA:.3f}  p_A_after={p_A_after:.3f}  '
                  f'A_displaced={A_displaced:.3f}  p_B_after={p_B_after:.3f}')
            with open(ROWS_FILE, 'w') as f: json.dump({'rows': rows}, f, indent=2)
        except Exception as e:
            print(f'[{run_idx:>2}/{total_runs}] FAILED: {type(e).__name__}: {e}')
            rows.append({'pair_idx': p_idx, 'condition': cond, 'status': 'failed', 'error': str(e)[:200]})
            torch.cuda.empty_cache()

elapsed = (datetime.now() - started).total_seconds() / 60
print(f'Total runtime: {elapsed:.1f} min')
restore(model, original_state)

### 11.7 Aggregate and test

In [ ]:
df = pd.DataFrame([r for r in rows if r.get('status') == 'ok'])
print(f'OK rows: {len(df)}')

agg = df.groupby('condition').agg(
    n=('pair_idx','count'),
    A_displaced_mean=('A_displaced','mean'),
    A_displaced_median=('A_displaced','median'),
    A_displaced_std=('A_displaced','std'),
    p_B_after_mean=('p_B_after_B','mean'),
).round(3)
print('\nPer-condition aggregate:')
print(agg)

print('\n=== Pairwise Wilcoxon (paired) ===')
from itertools import combinations
results = {}
for c1, c2 in combinations(CONDITIONS, 2):
    piv = df.pivot_table(index='pair_idx', columns='condition', values='A_displaced').dropna(subset=[c1, c2])
    if len(piv) >= 4:
        w, p = stats.wilcoxon(piv[c1], piv[c2])
        sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
        diff = piv[c1].mean() - piv[c2].mean()
        print(f'  {c1:>13} vs {c2:>13}: p={p:.4f} {sig}  (diff={diff:+.3f})')
        results[(c1,c2)] = {'p': float(p), 'diff': float(diff)}

print('\n=== Verdict ===')
m = {c: df[df['condition']==c]['A_displaced'].mean() for c in CONDITIONS}
for c in CONDITIONS:
    print(f'  {c:>13}: {m[c]:.3f}')

if m.get('dry_run', 0) > m.get('skip', 0) + 0.1:
    print('\n>>> NOVEL: even no-update gradient computation damages A.')
elif m.get('random_target', 0) > m.get('skip', 0) + 0.1 and m.get('random_target', 0) >= m.get('real', 0) - 0.1:
    print('\n>>> NOVEL: random-target edits as destructive as real edits. Content does NOT matter.')
elif m.get('real', 0) > m.get('random_target', 0) + 0.1:
    print('\n>>> Edit content matters: real edits more destructive than random-target edits.')
else:
    print('\n>>> Pattern unclear, see above.')

### 11.8 Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
order = ['skip','dry_run','random_target','real']
labels = ['C0: no edit B', 'C2: gradients (lr~0)', 'C3: random target', 'C1: real edit']
means = [df[df['condition']==c]['A_displaced'].mean() for c in order]
stds  = [df[df['condition']==c]['A_displaced'].std()  for c in order]
colors = ['#888888', '#cc6633', '#3366cc', '#cc3333']
bars = ax.bar(labels, means, yerr=stds, capsize=5, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylabel('A_displaced')
ax.set_title('Does editing B destroy A even when B fails?')
ax.grid(alpha=0.3, axis='y')
ax.set_ylim(-0.05, 1.05)
ax.tick_params(axis='x', rotation=15)
for bar, mv in zip(bars, means):
    ax.annotate(f'{mv:.2f}', (bar.get_x()+bar.get_width()/2, bar.get_height()+0.02),
                ha='center', fontsize=10, fontweight='bold')

ax = axes[1]
piv = df.pivot_table(index='pair_idx', columns='condition', values='A_displaced')
if 'skip' in piv.columns and 'real' in piv.columns:
    ax.scatter(piv['skip'], piv['real'], s=80, alpha=0.7, color='#cc3333', edgecolors='black', label='C0 vs C1 real')
if 'dry_run' in piv.columns and 'skip' in piv.columns:
    ax.scatter(piv['skip'], piv['dry_run'], s=80, alpha=0.7, color='#cc6633', edgecolors='black', label='C0 vs C2 dry_run')
ax.plot([0,1], [0,1], 'k--', alpha=0.5, label='y=x (no extra damage)')
ax.set_xlabel('A_displaced under no edit B (C0)')
ax.set_ylabel('A_displaced under attempted edit B')
ax.set_title('Per-pair: damage above no-edit baseline')
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / 'fig1_empty_edit.png', dpi=140); plt.show()

### 11.9 Save and bundle

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'notebook': 'Notebook 11 - Empty Edit Control', 'timestamp': ts,
    'model': MODEL_NAME, 'n_pairs': len(pairs), 'conditions': CONDITIONS,
    'data_source': 'pair indices auto-fetched from github.com/rukmini-17/targeted-unlearning',
    'per_condition_means': {c: float(df[df['condition']==c]['A_displaced'].mean()) for c in CONDITIONS},
    'per_condition_n': {c: int((df['condition']==c).sum()) for c in CONDITIONS},
    'pairwise_tests': {f'{a}_vs_{b}': v for (a,b), v in results.items()},
}
with open(RESULTS_DIR / f'summary_nb11_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
df.to_csv(RESULTS_DIR / f'df_nb11_{ts}.csv', index=False)
print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb11_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')